# Assignment 7 — Path Tracing

> **GAMES101 — Intro to Computer Graphics** (Lingqi Yan, UCSB).
> Course site: <https://sites.cs.ucsb.edu/~lingqi/teaching/games101.html>
>
> The course ships C++ starter code with Eigen + OpenCV. I'm doing the same tasks in
> Python notebooks so I can iterate on the math cell-by-cell. Notes at the top of each
> notebook are what I actually needed to remember to get the assignment out.

## Topic

The hardest assignment in the course. **Path tracing** is unbiased Monte-Carlo
solution of the rendering equation:

$$L_o(x, \omega_o) = L_e + \int_\Omega f_r(x, \omega_i, \omega_o) L_i(x, \omega_i) (\omega_i \cdot n) \, d\omega_i$$

Every pixel fires many rays; at each hit we split into:
- **Direct lighting** — sample the light source area explicitly, cast a shadow ray.
- **Indirect lighting** — pick a random direction on the hemisphere, recurse, terminate
  with **Russian roulette** so the average path length stays finite but unbiased.

The subtle part: if the direct term samples the light AND the indirect term happens
to hit the light, you double-count. Fix: indirect rays skip emitters.

The canonical test scene is the **Cornell box** — a red-and-green room with an area
light in the ceiling. It's a benchmark because it stresses colour bleeding, soft
shadows, and glossy reflection all at once.

![path tracing](https://upload.wikimedia.org/wikipedia/commons/e/e0/Path_tracing_001.png)
*Path-traced render (Wikipedia, public domain).*

![cornell box](https://upload.wikimedia.org/wikipedia/commons/9/92/Cornell_Box_Octane_%286K%2C_8bit%29.png)
*Cornell box, rendered in Octane. This is the target look. (Wikipedia)*

See: [Path tracing](https://en.wikipedia.org/wiki/Path_tracing), [Rendering equation](https://en.wikipedia.org/wiki/Rendering_equation), [Cornell box](https://en.wikipedia.org/wiki/Cornell_box).


In [1]:
import numpy as np

def cast_ray(scene, ray, depth):
    if depth > 5:
        return np.zeros(3)
    hit = scene.intersect(ray)
    if hit is None:
        return np.zeros(3)
    if hit.material.is_emissive:
        return hit.material.emit
    # sample the hemisphere uniformly -- yes it's inefficient
    wi = sample_hemisphere(hit.normal)
    pdf = 1 / (2 * np.pi)
    Li = cast_ray(scene, Ray(hit.pos, wi), depth + 1)
    brdf = hit.material.albedo / np.pi
    cos_theta = max(0.0, hit.normal @ wi)
    return Li * brdf * cos_theta / pdf


Renders but the Cornell box is basically black -- direct lighting almost never gets hit by uniform hemisphere sampling. Need explicit light sampling.